# Data Cleaning — Match

In [6]:
import pandas as pd

# Keep ALL matches — away results needed for form engineering
df = pd.read_csv('../../data/raw/gold_match.csv', sep=',', on_bad_lines='skip')
print(df.shape)
df.head()

(142, 27)


,match_id,match_date,kickoff_time_local,match_date_utc,kickoff_time_utc,matchday,competition_id,competition_name,season,season_id,...,venue,winner,result_home,goals_home_ht,goals_away_ht,goals_home_ft,goals_away_ft,is_home_match,last_result_vs_opponent,tickets_scanned
0,d0te6swsv2y99ywgugc2utbmc,2022-07-23,18:15:00,2022-07-23Z,16:15:00Z,1.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,Guldensporenstadion,away,L,0,0,0,2,False,L 1-2,NaN
1,d256yo3eng04m0fu7b4sl7wno,2022-07-30,18:15:00,2022-07-30Z,16:15:00Z,2.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,King Power at Den Dreef Stadion,home,W,1,0,2,0,True,NaN,5565.0
2,d3pqkck2grzx98jg4sofhp8us,2022-08-07,21:00:00,2022-08-07Z,19:00:00Z,3.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,Bosuilstadion,home,W,2,1,4,2,False,L 0-1,NaN
3,d4mn5ksbxuvnaww4pmommxhqs,2022-08-14,18:30:00,2022-08-14Z,16:30:00Z,4.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,King Power at Den Dreef Stadion,away,L,0,2,0,3,True,L 1-4,7440.0
4,d5htdqmc8w72upys41sfxhfkk,2022-08-21,18:30:00,2022-08-21Z,16:30:00Z,5.0,4zwgbb66rif2spcoeeol2motx,Belgian Jupiler Pro League,2022/2023,5wj8l9y4s3484k6puwup793pw,...,Stade Maurice Dufrasne,away,L,0,2,1,3,False,W 2-1,NaN


In [7]:
# Fix data types
df['match_date'] = pd.to_datetime(df['match_date'])
df['kickoff_time_local'] = pd.to_datetime(df['kickoff_time_local'], format='%H:%M:%S').dt.time
df['is_home_match'] = df['is_home_match'].map({'true': True, 'false': False}).astype(bool)
df['matchday'] = df['matchday'].astype('Int64')

In [8]:
# Parse last_result_vs_opponent (e.g. "L 1-2") into result letter and score string
df['last_h2h_result'] = df['last_result_vs_opponent'].str.extract(r'^([WDL])')
df['last_h2h_score']  = df['last_result_vs_opponent'].str.extract(r'([0-9]+-[0-9]+)$')
df = df.drop(columns=['last_result_vs_opponent'])

In [9]:
# Drop redundant, constant, and internal columns
cols_to_drop = [
    'match_date_utc',     # redundant with match_date
    'kickoff_time_utc',   # redundant with kickoff_time_local
    'competition_id',     # internal ID
    'competition_name',   # constant
    'season',             # covered by engineered features
    'season_id',          # internal ID
    'home_team_id',       # internal ID
    'away_team_id',       # internal ID
    'home_team_code',     # redundant with home_team
    'away_team_code',     # redundant with away_team
    'venue',              # constant for home games, irrelevant for away
    'winner',             # duplicate of result_home
    'home_team',          # always OHL for home games; away_team captures opponent
    'goals_home_ht',      # half-time scores — minimal attendance prediction value
    'goals_away_ht',
]
df = df.drop(columns=cols_to_drop)

# print(df.shape)
# print(df.dtypes)
df.head()

,match_id,match_date,kickoff_time_local,matchday,stage,away_team,result_home,goals_home_ft,goals_away_ft,is_home_match,tickets_scanned,last_h2h_result,last_h2h_score
0,d0te6swsv2y99ywgugc2utbmc,2022-07-23,18:15:00,1,Regular Season,OH Leuven,L,0,2,True,NaN,L,1-2
1,d256yo3eng04m0fu7b4sl7wno,2022-07-30,18:15:00,2,Regular Season,Westerlo,W,2,0,True,5565.0,NaN,NaN
2,d3pqkck2grzx98jg4sofhp8us,2022-08-07,21:00:00,3,Regular Season,OH Leuven,W,4,2,True,NaN,L,0-1
3,d4mn5ksbxuvnaww4pmommxhqs,2022-08-14,18:30:00,4,Regular Season,Club Brugge,L,0,3,True,7440.0,L,1-4
4,d5htdqmc8w72upys41sfxhfkk,2022-08-21,18:30:00,5,Regular Season,OH Leuven,L,1,3,True,NaN,W,2-1


In [10]:
# df.to_csv('../../data/cleaned/gold_match_cleaned.csv', index=False)
# print('Saved.')